# Deriving parameters based on radial profiles

In [ ]:
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from dwarfs.photometry import photometry as phot
from dwarfs.photometry import utils

### Building an isophotal profile

To derive things like concentration and half-life radius, we need a radial curve of growth.  The first step in getting these parameters is thus to determine the isophotal profile of the galaxy in question.  Below is a demonstration of how to do this using the software package included in this repository.

The example image used has been hand-masked of interloping sources, as well as one random pointlike source within the galaxy (for demonstration purposes).

In [ ]:
# Read in the masked image
imarr = fits.getdata('masked.fits')

# Convert it to a masked array
maskarr = np.ma.masked_array(imarr, mask=(imarr == -999))

In [ ]:
# Make a profile instance using the masked image
prof = phot.Profile(maskarr)

# Estimate the galaxy center
halfBoxWid = 20  # Determines the size of the box used in the center of the image for this estimate
x0, y0 = prof.initializeCenter(halfBoxWid)

# Show this on the image
lgIm = utils.ds9LogScale(maskarr)
plt.figure(figsize=(3, 3))
plt.imshow(lgIm, vmin=-3, vmax=1)
plt.plot(x0, y0, 'rx')

In [ ]:
# Now fit isophotes, fixing no parameters to see what happens
ellipse = prof.initializeEllipse(x0=x0, y0=y0, sma=20, pa=0, eps=0.5)
isoList = prof.fitEllipses(ellipse)  # Using default parameters for fitter

In [ ]:
# Showing these on the image
plt.figure(figsize=(4, 4))
prof.visualizeEllipses(isoList, vmin=-3, vmax=1)

In [ ]:
# Plotting some profiles
fig, ax = plt.subplots(2, 2, figsize=(8, 8))

ax[0, 0].plot(isoList.sma, np.degrees(isoList.pa), 'k.')
ax[0, 0].set_xlabel('Radius (px)')
ax[0, 0].set_ylabel('Position angle (degrees)')

ax[0, 1].plot(isoList.sma, isoList.eps, 'k.')
ax[0, 1].set_xlabel('Radius (px)')
ax[0, 1].set_ylabel('Ellipticity (1-b/a)')

ax[1, 0].plot(isoList.sma, isoList.x0, 'k.')
ax[1, 0].plot(0, x0, 'rx')  # The initial guess
ax[1, 0].set_xlabel('Radius (px)')
ax[1, 0].set_ylabel('Center x (px)')

ax[1, 1].plot(isoList.sma, isoList.y0, 'k.')
ax[1, 1].plot(0, y0, 'rx')  # The initial guess
ax[1, 1].set_xlabel('Radius (px)')
ax[1, 1].set_ylabel('Center y (px)')
plt.subplots_adjust(wspace=0.3, hspace=0.2)

Best practice when deriving a curve of growth is to keep the isophote shapes fixed, so we need some strategy to pick the best isophote parameters for this fixed shape.  For early type dwarfs, this should be straight-forward, since they should have fairly regular isophotes--maybe we pick the mean position angle and ellipticity of the whole profile, for example.  For irregular dwarfs like this one, it's harder.  Past studies have used the mean parameters derived in the far outskirts, where the profile shapes seem to converge to more constant values.  So here, for example, we can use the mean of the values beyond ~50px radius.  We'll do that for now, but evidently an automated means of picking these values is needed for the whole sample.

Doing this, we are choosing a center offset from the galaxy bulge, which is probably a bad idea.

In [ ]:
# Define basic parameters
want = isoList.sma > 50
x0 = np.mean(isoList.x0[want])
y0 = np.mean(isoList.y0[want])
pa = np.mean(isoList.pa[want])
eps = np.mean(isoList.eps[want])
print(x0, y0, pa, eps)

Now we can derive the profile using fixed parameters, which is better for things like half-light radius and concentration.  But first I'm going to interpolate the galaxy flux across the masks.  We need a noise estimate for this.

In [ ]:
# Getting estimates of the background noise and sky local to the galaxy
maxRad = isoList.sma[-1]*1.5  # For noise estimate; masks galaxy out to this radius
rms, sbLim, sky = phot.measureImageNoise(maskarr, x0, y0, maxRad, seed=12345)  # Using seed to keep results stable
print(rms, sbLim, sky)

In [ ]:
interparr = phot.interpolateAcrossMasks(maskarr, x0, y0, maxRad, pa, eps, rms, ampCut=0.01)

lgIm = utils.ds9LogScale(interparr)
plt.figure(figsize=(3, 3))
plt.imshow(lgIm, vmin=-3, vmax=1)

Now we redo the isophote measurements with fixed ellipses on this "unmasked" image.  We do this by exploiting the software; it has a parameter called 'maxrit', which is the maximum radius out to which to try fitting the isophote shapes.  If we set that to 1 pixel, it won't fit at all.  When doing this, though, always set a maximum radius for the profile (maxsma), or the fitting algorithm will hang.

I'm also going to swap to a linear spacing (fitter default is logarithmic, to enhance S/N in the outskirts), with a width of 0.5" (3 pixels), which is maybe sort of close to the seeing in this image.

In [ ]:
prof = phot.Profile(interparr)
ellipse = prof.initializeEllipse(x0=x0, y0=y0, sma=20, pa=pa, eps=eps)
isoList = prof.fitEllipses(ellipse, maxsma=isoList.sma[-1], maxrit=1, step=3, linear=True)

plt.figure(figsize=(4, 4))
prof.visualizeEllipses(isoList, vmin=-3, vmax=1)

In [ ]:
# Curve of growth and surface brightness profile
magZp = 27  # From HSC documentation
pxScale = 0.17
rad = isoList.sma * pxScale
mag = -2.5*np.log10(isoList.tflux_e) + magZp
sb = -2.5*np.log10(isoList.intens) + magZp + 2.5*np.log10(pxScale**2)

fig, ax = plt.subplots(1, 2, figsize=(8, 3))

ax[0].plot(rad, mag, 'k.')
ax[0].set_xlabel('Radius (arcsec)')
ax[0].set_ylabel(r'$m(<R)$ (mag)')
ax[0].set_ylim([25, 18])

ax[1].plot(rad, sb, 'k.')
ax[1].set_xlabel('Radius (arcsec)')
ax[1].set_ylabel(r'$\mu$ (mag arcsec$^{-2}$)')
ax[1].set_ylim([31.5, 22.5])

Note the rise in the center of the surface brightness profile: that's where the isophotes encounter the bulge for the first time.  The total magnitude should be insensitive to this, but other parameters might not be.

### Deriving morphological parameters from these

In [ ]:
# Create an instance of a new class using the isoList in table format
isoTab = prof.toTable(isoList)
dervals = phot.DerivedValues(isoTab, magZp=magZp, pxScale=pxScale)

In [ ]:
# Derive curve of growth corrected for local sky (see above)
sma, cog = dervals.curveOfGrowth(sky=sky)
# Use this to derive the total magnitude
totmag = dervals.totalMagnitude(sma, cog)
print(totmag)

We also need some uncertainty estimates on these values.  This comes from two places: photometric uncertainty, and systematic uncertainty.

The former is the usual contribution from Poisson uncertainty in the number of source counts, Poisson uncertainty in the number of sky background counts, read noise, and dark current.  See: http://spiff.rit.edu/classes/ast613/lectures/signal/signal_illus.html

All of these uncertainties need to come in units of electrons, then converted from that back to magnitudes.  So you will need first to convert from whatever units your stamps are in into ADU, then from ADU into electrons via the camera's gain--since HSC is composed of multiple CCDs with multiple amplifiers, the "gain" isn't a single number, but there might be one average value of it you can use for this.  Theoretically you can find that info for HSC images somewhere in their vast quantities of documentation.  If not, you can always e-mail someone, I guess.  Probably they've gotten this question before.

The systematic uncertainty is mostly the uncertainty inherent to the local sky estimation, which causes the curve of growth and the surface brightness profile to wiggle around in the outskirts.  This is easier: we can estimate that by just perturbing the curves and recalculating the desired values each time, then taking the standard deviation of those perturbed values.

In [ ]:
# Derive uncertainty due to sky estimate
# N random perturbations to the local sky centered at the mean value, rederiving magnitude each time
N = 1000
rng = np.random.default_rng(12345)  # Using seed to keep results stable
randskies = rng.normal(sky, sbLim, size=N)  # Assuming sky obeys Gaussian statistics
randmags = []
for i in range(N):
    sma, cog = dervals.curveOfGrowth(sky=randskies[i])
    randmag = dervals.totalMagnitude(sma, cog)
    randmags.append(randmag)
magerr = np.nanstd(randmags)
print('Uncertainty on magnitude from sky subtraction: %.4f magnitudes' % (magerr))

Other quantities depend on the derived total magnitude, so save your equivalent `randmags` list or array and calculate these derived parameters from that to estimate the sky uncertainty on those values.

In [ ]:
# Use this to derive other quantities
reff = dervals.fractionalRadius(totmag, fluxFrac=0.5)  # Half-light radius
c82 = dervals.concentration82(totmag)  # C82 = 5log(R80/R20)
cRe = dervals.concentrationRe(totmag)  # From Trujillo et al. (2001)
rpetro = dervals.petrosianRadius()
print('Half-light radius = %.4f arcsec' % (reff*pxScale))
print('C82 = %.4f' % (c82))
print('C (Trujillo) = %.4f' % (cRe))
print('Petrosian radius = %.4f arcsec' % (rpetro*pxScale))

Concentration parameter is close to what it should be for an n=1 profile, which is good.  Petrosian radius is about twice half-light radius, which is also true for n=1 profiles.  So everything here seems to be in order.

### An attempt at automating the ellipse-fitting procedure

When `prof.fitEllipses` fails, typically because the starting parameters you fed it were bogus, it sends a warning but doesn't crash your script.  This typically results in an empty dictionary of fit parameters for that object.  Since there are so few objects (~300 out to z=0.08), you can run the fitter individually on each, adjusting the starting parameters by hand until it works, but instead of that, here's a way to semi-automate the process.

We'll try looping over different ellipticity and position angle values until the fitter actually fits the profile.  We can use the center finding output for the center estimate, which should be pretty accurate on its own (maybe better than the one the profile fitter locates, honestly).  Depending on how sensitive the fitter is to these starting parameters, you can add more resolution in ellipticity or position angle, or you can add another nested loop over starting semi-major axis (which is currently fixed at 20px).

In [ ]:
ellips = np.array([0.5, 0.3, 0.8])  # Try halfway, then either end
pas = np.arange(0, 180, 45)  # 180 is redundant with 0
print(ellips)
print(pas)

In [ ]:
# Looping until it works.  Ignore the warning messages.
flag = 0
for i in range(len(pas)):
    for j in range(len(ellips)):
        ellipse = prof.initializeEllipse(x0=x0, y0=y0, sma=20, pa=pas[i], eps=ellips[j])
        isoList = prof.fitEllipses(ellipse)  # Using default parameters for fitter
        if len(isoList.to_table()) == 0:
            continue
        else:
            flag = 1
            print('Fit succeeded')
            break
    if flag:
        break

In [ ]:
print(len(isoList.to_table()))

The length here is not zero, so we have a real table.  Whether this works consistently or not will, of course, require a trial run on the whole set of stamps.